In [ ]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print("Downloaded to:", path)

In [ ]:
import os
import pandas as pd

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
print("CSV files found:", csv_candidates)

df = pd.read_csv(csv_candidates[0])
print()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df.head(3).to_string())

In [ ]:
def find_column(df, keywords):
    matches = [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]
    if not matches:
        raise ValueError(f"No column found matching {keywords} among {df.columns.tolist()}")
    if len(matches) > 1:
        print(f"WARNING: multiple columns matched {keywords}: {matches} — using the first one, verify this is right")
    return matches[0]

question_col = find_column(df, ["question"])
answer_col = find_column(df, ["answer"])
print("Question column:", question_col)
print("Answer column:", answer_col)

print()
print("Nulls in question:", df[question_col].isna().sum())
print("Nulls in answer:", df[answer_col].isna().sum())

df = df.dropna(subset=[question_col, answer_col]).reset_index(drop=True)
print("After dropping nulls:", df.shape)

In [ ]:
!pip install -q nltk

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

import re
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

_stemmer = PorterStemmer()
_stop_words = set(stopwords.words("english"))

def stem_and_clean(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    tokens = word_tokenize(text)
    stemmed = [_stemmer.stem(t) for t in tokens if len(t) > 1 and t not in _stop_words]
    return " ".join(stemmed)

# Sanity check the exact bug this fixes, before trusting it on the real corpus
test = stem_and_clean("How do I treat my migraines?")
print("Preprocessed test string:", repr(test))

In [ ]:
questions = df[question_col].astype(str).tolist()
answers = df[answer_col].astype(str).tolist()

preprocessed_questions = [stem_and_clean(q) for q in questions]

print("Example before/after:")
for i in [0, 1, 2]:
    print(f"  original:     {questions[i]!r}")
    print(f"  preprocessed: {preprocessed_questions[i]!r}")
    print()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# IMPORTANT: fit on the already-preprocessed text with TfidfVectorizer's
# OWN default tokenizer — never pass tokenizer=stem_and_clean here. A
# vectorizer fit with a custom Python callable can't be safely loaded back
# in a different process (the service) — confirmed this fails with
# AttributeError on load. Pre-stemming the text avoids the issue entirely.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_questions)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def test_retrieve(query, top_k=3):
    cleaned = stem_and_clean(query)
    query_vec = vectorizer.transform([cleaned])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    for i in top_idx:
        print(f"  score={sims[i]:.3f}  Q: {questions[i]}")

print("Test 1: plural/verb-form robustness")
test_retrieve("How do I treat my migraines?")

print()
print("Test 2: a completely different real question")
test_retrieve("What are the symptoms of diabetes?")

print()
print("Test 3: off-topic — expect low scores across the board")
test_retrieve("what is the capital of France")

In [ ]:
qa_corpus = [
    {"question": questions[i], "answer": answers[i], "sources": ["MedQuAD"]}
    for i in range(len(questions))
]
print(f"{len(qa_corpus)} entries ready for export")
print(qa_corpus[0])

In [ ]:
import joblib
import json
import os

os.makedirs("chatbot_artifacts", exist_ok=True)

joblib.dump(vectorizer, "chatbot_artifacts/tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, "chatbot_artifacts/tfidf_matrix.joblib")
with open("chatbot_artifacts/qa_corpus.json", "w") as f:
    json.dump(qa_corpus, f)

print("Artifacts written:")
!ls -la chatbot_artifacts

In [ ]:
import shutil
shutil.make_archive("v1_chatbot_artifacts", "zip", "chatbot_artifacts")

from google.colab import files
files.download("v1_chatbot_artifacts.zip")